# Topic 29 — PyTorch Fundamentals
### Theory → tensors → autograd → Dataset/DataLoader → nn.Module → full training loop.

Everything you hand-built from scratch in Topics 27-28 (forward pass, backprop, gradient descent,
Adam) is exactly what PyTorch automates for you. This notebook rebuilds the SAME XOR problem from
Topic 27, but using PyTorch's tools — so you can directly compare "by hand" vs "with the framework".

**Colab tip**: Runtime -> Change runtime type -> select a GPU (T4) if you want to try `.to("cuda")`
below, though none of this notebook's tiny examples actually need one.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

print("PyTorch version:", torch.__version__)
print("CUDA (GPU) available:", torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"
print("using device:", device)

## 1. Tensors — PyTorch's version of NumPy arrays

A `Tensor` is PyTorch's core data structure — almost identical to a NumPy array (Topic 1), but it
can live on a GPU and supports automatic differentiation (next section).

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0])
print("tensor:", x, " shape:", x.shape, " dtype:", x.dtype)

# Converting between NumPy and PyTorch
np_array = np.array([[1, 2], [3, 4]])
tensor_from_np = torch.from_numpy(np_array)
back_to_np = tensor_from_np.numpy()
print("\nnumpy -> tensor -> numpy:", back_to_np)

# Moving a tensor to GPU (if available) -- ".to(device)" is the standard pattern
x_on_device = x.to(device)
print("\ntensor device:", x_on_device.device)

# Familiar operations, same names as NumPy
a = torch.tensor([[1., 2.], [3., 4.]])
b = torch.tensor([[5., 6.], [7., 8.]])
print("\nmatmul (a @ b):\n", a @ b)
print("transpose (a.T):\n", a.T)
print("reshape:", torch.arange(6).reshape(2, 3))

## 2. Autograd — automatic differentiation

Set `requires_grad=True` on a tensor and PyTorch tracks every operation done to it, building a
computation graph. Calling `.backward()` then computes ALL gradients automatically — this
literally replaces the manual backprop math you wrote by hand in Topic 27.

In [ ]:
w = torch.tensor(3.0, requires_grad=True)   # a parameter we want to learn
x_val = torch.tensor(2.0)

y = w * x_val + 1        # y = 3*2 + 1 = 7 (a tiny "forward pass")
loss = (y - 10) ** 2      # squared error against a target of 10

print("y =", y.item(), " loss =", loss.item())

loss.backward()           # autograd computes d(loss)/dw automatically
print("gradient dw:", w.grad.item())
# Verify by hand: loss = (3wx+1-10)^2 -> d(loss)/dw = 2*(3wx+1-10)*3x = 2*(7-10)*6 = -36. Matches!

## 3. `Dataset` and `DataLoader`

`Dataset` wraps your data with a standard interface (`__len__`, `__getitem__`).
`DataLoader` wraps a `Dataset` and automatically handles batching, shuffling, and iteration —
this is the practical implementation of Topic 28's "batch/mini-batch/epoch" concepts.

In [ ]:
class XORDataset(Dataset):
    def __init__(self):
        self.X = torch.tensor([[0,0],[0,1],[1,0],[1,1]], dtype=torch.float32)
        self.y = torch.tensor([[0],[1],[1],[0]], dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

dataset = XORDataset()
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

for batch_X, batch_y in dataloader:
    print("batch X:\n", batch_X, "\nbatch y:\n", batch_y, "\n---")
# Each iteration of this loop is one "iteration/step" from Topic 28's vocabulary.

## 4. `nn.Module` — defining a network

Every PyTorch model subclasses `nn.Module`. You define the layers in `__init__`, then describe
how data flows through them in `forward()`. This is the same 2-input -> hidden -> 1-output
architecture from Topic 27, but built with PyTorch's building blocks.

In [ ]:
class SimpleNet(nn.Module):
    def __init__(self, n_input=2, n_hidden=4, n_output=1):
        super().__init__()
        self.layer1 = nn.Linear(n_input, n_hidden)   # a "Linear" layer = the Wx+b you built by hand
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(n_hidden, n_output)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        x = self.sigmoid(x)
        return x

model = SimpleNet().to(device)
print(model)

# Count trainable parameters
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("\ntrainable parameters:", n_params)
# (2*4 + 4) + (4*1 + 1) = 12 + 5 = 17 -- matches n_input*n_hidden + n_hidden biases, etc.

## 5. Loss functions & optimizers

In [ ]:
criterion = nn.BCELoss()                          # binary cross-entropy (Topic 27)
optimizer = optim.Adam(model.parameters(), lr=0.05)   # Adam (Topic 28), applied to ALL model params

# A single manual step, to see the pieces before the full loop
X_batch, y_batch = dataset.X.to(device), dataset.y.to(device)

optimizer.zero_grad()             # clear old gradients (they accumulate by default!)
predictions = model(X_batch)      # forward pass
loss = criterion(predictions, y_batch)
loss.backward()                   # backward pass -- autograd computes every gradient
optimizer.step()                  # apply the gradient update to every parameter

print("loss after one step:", loss.item())

## 6. The full training loop

In [ ]:
model = SimpleNet().to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.05)

n_epochs = 300
loss_history = []

model.train()   # sets the model to "training mode" (matters for dropout/batchnorm, Topics 31-32)
for epoch in range(n_epochs):
    epoch_losses = []
    for batch_X, batch_y in dataloader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        optimizer.zero_grad()
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()

        epoch_losses.append(loss.item())
    loss_history.append(np.mean(epoch_losses))

plt.figure(figsize=(5, 4))
plt.plot(loss_history)
plt.xlabel("epoch"); plt.ylabel("loss")
plt.title("PyTorch training loop on XOR")
plt.show()

## 7. Evaluation mode & inference

`.eval()` switches off training-only behaviors (dropout, batchnorm updates — Topic 31-32).
`torch.no_grad()` disables gradient tracking during inference, since you don't need gradients
when you're not training — this saves memory and computation.

In [ ]:
model.eval()
with torch.no_grad():
    final_preds = model(dataset.X.to(device))
print("final predictions:\n", final_preds.cpu().numpy().round(3))
print("true labels:\n", dataset.y.numpy().flatten())

## 8. Model saving & loading (preview of Topic 42)

In [ ]:
torch.save(model.state_dict(), "xor_model.pt")

loaded_model = SimpleNet().to(device)
loaded_model.load_state_dict(torch.load("xor_model.pt"))
loaded_model.eval()

with torch.no_grad():
    loaded_preds = loaded_model(dataset.X.to(device))
print("loaded model predictions match original:",
      torch.allclose(final_preds, loaded_preds))

## Exercise

In [ ]:
# --- Try it yourself ---
# 1. Change nn.ReLU() to nn.Tanh() in SimpleNet and retrain -- compare the final loss.
# 2. Change DataLoader's batch_size to 1 (full SGD, Topic 28) and to 4 (full-batch, since there are
#    only 4 samples) -- compare loss curves.
# 3. Add a second hidden layer (nn.Linear(n_hidden, n_hidden) + nn.ReLU()) to SimpleNet.
# 4. Compare model.parameters() count before/after your architecture change -- confirm your
#    mental math about parameter counts matches what PyTorch reports.

---
### Next up: **Topic 30 — Build an MLP** (a full multilayer perceptron on real-ish tabular/text data).

Say "next" when you're ready.